# Loss
作用：
- 计算实际输出和目标之间的关系
- 为更新输出提供依据（反向传播）
## L1Loss

In [7]:
import torch
from torch import nn
from torch.nn import L1Loss

inputs = torch.tensor([1, 2, 3], dtype=torch.float32)
targets = torch.tensor([1, 2, 5], dtype=torch.float32)

inputs = torch.reshape(inputs, (1, 1, 1, 3))
targets = torch.reshape(targets, (1, 1, 1, 3))

loss = L1Loss()
# loss = L1Loss(reduction="sum")
res = loss(inputs, targets)
print(res)

tensor(0.6667)


## MSELoss

平方差损失函数

In [5]:
from torch.nn import MSELoss

mseLoss = MSELoss()
res = mseLoss(inputs, targets)
print(res)

tensor(1.3333)


## CrossEntropLoss

交叉商

In [2]:
import torch
from torch import nn
crossLoss = nn.CrossEntropyLoss()
res = crossLoss(torch.tensor([0.1, 0.2, 0.3]), torch.tensor([1]))
print(res)

tensor(1.1019)


In [11]:
import torch
import torchvision
from torch.utils.data import DataLoader
from torch import nn
from torch.nn import Conv2d, MaxPool2d, Flatten, Linear, Sequential
from torch.utils.tensorboard import SummaryWriter

dataset = torchvision.datasets.CIFAR10(root="/workspace/data", train=False, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = DataLoader(dataset, batch_size=64, drop_last = True)

# 使用 Sequential 简化网络书写
class CIFAR(nn.Module):
    def __init__(self):
        super(CIFAR, self).__init__()
        self.sequential = Sequential(
            Conv2d(3, 32, 5, padding = 2),
            MaxPool2d(2),
            Conv2d(32, 32, 5, padding = 2),
            MaxPool2d(2),
            Conv2d(32, 64, 5, padding = 2),
            MaxPool2d(2),
            Flatten(),
            Linear(1024, 64),
            Linear(64, 10)
        )

    def forward(self, input):
        return self.sequential(input)

crossLoss = nn.CrossEntropyLoss()
cifar = CIFAR()
# 优化器定义
optim = torch.optim.SGD(cifar.parameters(), lr = 0.01)
for epoch in range(10):
    running_loss = 0.0
    for data in dataloader:
        imgs, targets = data
        output = cifar(imgs)
        res = crossLoss(output, targets)

        # print("output", output)
        # print("targets: ", targets)
        # print("Loss: ", res)

        optim.zero_grad() # 梯度清零
        res.backward() # 反向传播 计算出梯度
        optim.step() # 根据梯度对每个参数进行调优

        running_loss = running_loss + res
    print("running loss", running_loss)

running loss tensor(358.4370, grad_fn=<AddBackward0>)
running loss tensor(355.2126, grad_fn=<AddBackward0>)
running loss tensor(342.5416, grad_fn=<AddBackward0>)
running loss tensor(320.7087, grad_fn=<AddBackward0>)
running loss tensor(310.1443, grad_fn=<AddBackward0>)
running loss tensor(301.2839, grad_fn=<AddBackward0>)
running loss tensor(291.9552, grad_fn=<AddBackward0>)
running loss tensor(284.9174, grad_fn=<AddBackward0>)
running loss tensor(277.7422, grad_fn=<AddBackward0>)
running loss tensor(270.8398, grad_fn=<AddBackward0>)


### 反向传播
作用: PyTorch 会自动使用链式法则，从 loss 开始，反向追溯整个计算图，计算出损失对每一个模型参数（权重）的梯度。
结果: 这个梯度值（一个和权重张量形状相同的张量）会被自动存储在每个参数的 .grad 属性里。比如 model.linear1.weight.grad 就会被填充上具体的梯度值。
对应比喻: 这就是“感受脚下坡度”的过程。它告诉我们，每个权重应该朝着哪个方向调整，才能让损失增加得最快。

### 为什么设置批次？

- 投票更准：利用平均值过滤掉个别数据的干扰。
- 跑得更快：让 GPU 的几千个核心同时干活，别闲着。
- 学得更稳：通过每组数据的微小差异，逼迫模型去寻找真正的本质特征（比如猫的耳朵、狗的鼻子），而不是死记硬背单张图片。

当我们给模型输入一个 Batch（比如 64 张图片）时，程序内部会发生以下事情：

分别计算：程序会计算这 64 张图中每一张图对应的梯度（即：每张图都告诉模型权重应该怎么改）。
取平均值：模型并不会立刻根据第一张图就调整权重，而是把这 64 个建议全部加起来，求一个平均值。
统一更新：最后，模型根据这个“平均意见”迈出一步。
为什么这能保证学习？

消除噪音：虽然这一批里有猫、有狗、有模糊的图片、有清晰的图片，但它们都有一个共同的目标——识别物体。那些“错误”或者“极端”的个体（比如一张长得像猫的狗）给出的偏差建议，会被其他几十张正常图片的正确建议抵消掉。
寻找共性：这个平均后的梯度，代表了这一组数据中体现出来的共性特征。模型通过学习共性，才能具备“举一反三”的泛化能力，而不是死记硬背某一张图。

